In [2]:
import os
import warnings
import numpy as np
import mne
import joblib

from mne.decoding import CSP

In [32]:
# ==========================================================
# CONFIGURATION
# ==========================================================

EPOCH_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/epochs"

FEATURE_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/CSP_leftright_subject_normalized"   

SUBJECT_EXPERIMENTS = [10, 20, 50, 109]

RUNS = ["R04", "R08", "R12"]

LEFT_EVENT = 2
RIGHT_EVENT = 3

N_COMPONENTS = 8

TRAIN_RATIO = 0.7

EXPECTED_TIMEPOINTS = 641

os.makedirs(FEATURE_DIR, exist_ok=True)

In [33]:
def load_subject_epochs(subject_list):

    """
    Loads R04, R08 and R12 epochs for each subject.

    Subjects having inconsistent epoch lengths are skipped.
    """

    subject_epochs = {}

    skipped_subjects = []

    for subject in subject_list:

        subject_folder = os.path.join(EPOCH_DIR, subject)

        epochs_list = []

        valid_subject = True

        for run in RUNS:

            file_path = os.path.join(
                subject_folder,
                f"{subject}{run}_epo.fif"
            )

            if not os.path.exists(file_path):
                continue

            try:

                epochs = mne.read_epochs(
                    file_path,
                    preload=True,
                    verbose=False
                )

                # -------------------------------------------------
                # Validate epoch length
                # -------------------------------------------------

                if epochs.get_data().shape[-1] != EXPECTED_TIMEPOINTS:

                    raise ValueError(
                        f"{run} has {epochs.get_data().shape[-1]} samples "
                        f"(expected {EXPECTED_TIMEPOINTS})"
                    )

                epochs_list.append(epochs)

            except Exception as e:

                print(f"Skipping {subject} ({run}) : {e}")

                valid_subject = False

                break

        if not valid_subject:
            skipped_subjects.append(subject)
            continue

        if len(epochs_list) == 0:
            skipped_subjects.append(subject)
            continue

        # Suppress harmless annotation warning
        with warnings.catch_warnings():

            warnings.filterwarnings(
                "ignore",
                message="Concatenation of Annotations within Epochs is not supported yet.*"
            )

            subject_epochs[subject] = mne.concatenate_epochs(
                epochs_list
            )

    print()

    print("=" * 60)

    print(f"Loaded Subjects : {len(subject_epochs)}")

    print(f"Skipped Subjects: {len(skipped_subjects)}")

    if len(skipped_subjects):

        print(skipped_subjects)

    print("=" * 60)

    return subject_epochs

In [34]:
def extract_left_right(subject_epochs):

    """
    Extract Left and Right imagery epochs and apply
    per-subject, per-channel z-score normalization.

    For each subject and channel:

        mean = calculated across epochs and timepoints
        std  = calculated across epochs and timepoints

        X_normalized = (X - mean) / std

    Labels:
        Left  -> 0
        Right -> 1
    """

    subject_data = {}

    for subject, epochs in subject_epochs.items():

        # --------------------------------------------------
        # Keep only Left and Right epochs
        # --------------------------------------------------

        mask = np.isin(
            epochs.events[:, -1],
            [LEFT_EVENT, RIGHT_EVENT]
        )

        selected_epochs = epochs[mask]

        X = selected_epochs.get_data()

        y = selected_epochs.events[:, -1]

        # --------------------------------------------------
        # Convert labels
        # --------------------------------------------------

        y = np.where(
            y == LEFT_EVENT,
            0,
            1
        )

        # --------------------------------------------------
        # Check both classes are present
        # --------------------------------------------------

        if len(np.unique(y)) != 2:

            print(
                f"Skipping {subject}: "
                f"only one class present"
            )

            continue

        # --------------------------------------------------
        # Per-subject, per-channel normalization
        # --------------------------------------------------
        #
        # X shape:
        # (epochs, channels, timepoints)
        #
        # axis=(0, 2):
        # calculate statistics across:
        #     - all epochs
        #     - all timepoints
        #
        # but separately for each channel
        # --------------------------------------------------

        subject_mean = X.mean(
            axis=(0, 2),
            keepdims=True
        )

        subject_std = X.std(
            axis=(0, 2),
            keepdims=True
        )

        # Prevent division by zero
        subject_std[
            subject_std < 1e-12
        ] = 1.0

        X_normalized = (
            X - subject_mean
        ) / subject_std

        # --------------------------------------------------
        # Store subject separately
        # --------------------------------------------------

        subject_data[subject] = (
            X_normalized,
            y
        )

    return subject_data

In [35]:
def prepare_train_test(subject_data):

    subjects = sorted(subject_data.keys())

    n_subjects = len(subjects)

    n_train = int(TRAIN_RATIO * n_subjects)

    train_subjects = subjects[:n_train]

    test_subjects = subjects[n_train:]

    X_train = []
    y_train = []

    X_test = []
    y_test = []

    # ------------------------
    # Training
    # ------------------------

    for subject in train_subjects:

        X, y = subject_data[subject]

        X_train.append(X)

        y_train.append(y)

    # ------------------------
    # Testing
    # ------------------------

    for subject in test_subjects:

        X, y = subject_data[subject]

        X_test.append(X)

        y_test.append(y)

    X_train = np.concatenate(X_train, axis=0)

    y_train = np.concatenate(y_train, axis=0)

    X_test = np.concatenate(X_test, axis=0)

    y_test = np.concatenate(y_test, axis=0)

    return (

        X_train,
        y_train,

        X_test,
        y_test,

        train_subjects,
        test_subjects

    )

In [36]:
# ==========================================================
# CELL 6 — VERIFY DATA + NORMALIZATION
# ==========================================================

subjects = [f"S{i:03d}" for i in range(1, 11)]

subject_epochs = load_subject_epochs(subjects)

# Normalization happens INSIDE this function
subject_data = extract_left_right(subject_epochs)

(
    X_train,
    y_train,
    X_test,
    y_test,
    train_subjects,
    test_subjects
) = prepare_train_test(subject_data)


# ==========================================================
# DATASET CHECK
# ==========================================================

print("=" * 60)

print("Train Subjects:")
print(train_subjects)

print("\nTest Subjects:")
print(test_subjects)

print("\n" + "=" * 60)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print("\nTraining Labels")
print("Left :", np.sum(y_train == 0))
print("Right:", np.sum(y_train == 1))

print("\nTesting Labels")
print("Left :", np.sum(y_test == 0))
print("Right:", np.sum(y_test == 1))


# ==========================================================
# NORMALIZATION CHECK
# ==========================================================

print("\n" + "=" * 60)
print("NORMALIZATION CHECK")
print("=" * 60)

first_subject = train_subjects[0]

X_check, _ = subject_data[first_subject]

channel_means = X_check.mean(axis=(0, 2))
channel_stds = X_check.std(axis=(0, 2))

print("Subject:", first_subject)

print(
    "Mean range:",
    channel_means.min(),
    "to",
    channel_means.max()
)

print(
    "Std range :",
    channel_stds.min(),
    "to",
    channel_stds.max()
)

Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 10
Skipped Subjects: 0
Train Subjects:
['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007']

Test Subjects:
['S008', 'S009', 'S010']

Training Shape : (315, 64, 641)
Testing Shape  : (135, 64, 641)

Training

In [37]:
def extract_csp_features(
    X_train,
    y_train,
    X_test
):
    """
    Learn CSP filters from training data
    and transform both train and test.
    """

    csp = CSP(
        n_components=N_COMPONENTS,
        reg=None,
        log=True,
        norm_trace=False
    )

    X_train_csp = csp.fit_transform(
        X_train,
        y_train
    )

    X_test_csp = csp.transform(
        X_test
    )

    return X_train_csp, X_test_csp, csp

In [38]:
def save_features(
    experiment_name,
    X_train,
    y_train,
    X_test,
    y_test,
    csp
):

    save_dir = os.path.join(
        FEATURE_DIR,
        experiment_name
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    np.save(
        os.path.join(save_dir, "X_train.npy"),
        X_train
    )

    np.save(
        os.path.join(save_dir, "y_train.npy"),
        y_train
    )

    np.save(
        os.path.join(save_dir, "X_test.npy"),
        X_test
    )

    np.save(
        os.path.join(save_dir, "y_test.npy"),
        y_test
    )

    # Save trained CSP object
    joblib.dump(
        csp,
        os.path.join(save_dir, "csp.pkl")
    )

    print(f"Saved -> {save_dir}")

In [39]:
# ==========================================================
# VALIDATE SUBJECT EPOCH LENGTHS
# ==========================================================

EXPECTED_TIMEPOINTS = 641


def get_valid_subjects(subjects):

    valid_subjects = []
    skipped_subjects = []

    print("\nChecking epoch lengths...")

    for subject in subjects:

        try:
            # Load only this subject
            subject_epochs = load_subject_epochs([subject])

            # Extract the subject's epochs from the returned structure
            epochs = subject_epochs[subject]

            X = epochs.get_data()

            print(
                f"{subject}: "
                f"shape={X.shape}"
            )

            # ----------------------------------------------
            # Check time dimension
            # ----------------------------------------------

            if X.shape[-1] != EXPECTED_TIMEPOINTS:

                print(
                    f"  SKIPPED {subject}: "
                    f"got {X.shape[-1]} samples, "
                    f"expected {EXPECTED_TIMEPOINTS}"
                )

                skipped_subjects.append(subject)

                continue

            valid_subjects.append(subject)

        except Exception as e:

            print(
                f"  SKIPPED {subject}: {e}"
            )

            skipped_subjects.append(subject)

    print("\n" + "-" * 60)

    print(
        f"Valid Subjects   : "
        f"{len(valid_subjects)}/{len(subjects)}"
    )

    print(
        f"Skipped Subjects : "
        f"{len(skipped_subjects)}"
    )

    if skipped_subjects:

        print(
            "Skipped List     :",
            skipped_subjects
        )

    print("-" * 60)

    return valid_subjects, skipped_subjects

In [40]:
for n_subjects in SUBJECT_EXPERIMENTS:

    print("\n" + "=" * 70)
    print(f"Processing {n_subjects} Subjects")
    print("=" * 70)

    subjects = [
        f"S{i:03d}"
        for i in range(1, n_subjects + 1)
    ]

    # ======================================================
    # STEP 1: CHECK EPOCH LENGTHS
    # ======================================================

    valid_subjects, skipped_subjects = get_valid_subjects(
        subjects
    )

    if len(valid_subjects) == 0:

        print(
            f"No valid subjects for "
            f"{n_subjects}-subject experiment."
        )

        continue

    # ======================================================
    # STEP 2: LOAD ONLY VALID SUBJECTS
    # ======================================================

    subject_epochs = load_subject_epochs(
        valid_subjects
    )

    # ======================================================
    # STEP 3: EXTRACT LEFT / RIGHT
    # ======================================================

    subject_data = extract_left_right(
        subject_epochs
    )

    # ======================================================
    # STEP 4: SUBJECT-LEVEL TRAIN / TEST SPLIT
    # ======================================================

    (
        X_train,
        y_train,
        X_test,
        y_test,
        train_subjects,
        test_subjects
    ) = prepare_train_test(
        subject_data
    )

    # ======================================================
    # STEP 5: CSP
    # ======================================================

    X_train_csp, X_test_csp, csp = extract_csp_features(
        X_train,
        y_train,
        X_test
    )

    # ======================================================
    # STEP 6: SAVE FEATURES
    # ======================================================

    save_features(
        f"{n_subjects}subjects",
        X_train_csp,
        y_train,
        X_test_csp,
        y_test,
        csp
    )

    # ======================================================
    # SUMMARY
    # ======================================================

    print("\n" + "-" * 60)

    print(f"Requested Subjects : {n_subjects}")
    print(f"Valid Subjects     : {len(valid_subjects)}")
    print(f"Skipped Subjects   : {len(skipped_subjects)}")

    if skipped_subjects:
        print(f"Skipped List       : {skipped_subjects}")

    print()

    print("Train Subjects :", len(train_subjects))
    print("Test Subjects  :", len(test_subjects))

    print()

    print("Train Epochs :", len(y_train))
    print("Test Epochs  :", len(y_test))

    print()

    print("Train Left  :", np.sum(y_train == 0))
    print("Train Right :", np.sum(y_train == 1))

    print()

    print("Test Left   :", np.sum(y_test == 0))
    print("Test Right  :", np.sum(y_test == 1))

    print()

    print(
        "Feature Shape (Train):",
        X_train_csp.shape
    )

    print(
        "Feature Shape (Test) :",
        X_test_csp.shape
    )

    print("-" * 60)


Processing 10 Subjects

Checking epoch lengths...
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 1
Skipped Subjects: 0
S001: shape=(90, 64, 641)
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 1
Skipped Subjects: 0
S002: shape=(90, 64, 641)
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 1
Skipped Subjects: 0
S003: shape=(90, 64, 641)
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 1
Skipped Subjects: 0
S004: shape=(90, 64, 641)
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 1
Skipped Subjects: 0
S005: shape=(90, 64, 641)
Not setting metadata
90 matching events found
No baseline correction applied

Loaded Subjects : 1
Skipped Subjects: 0
S006: shape=(90, 64, 641)
Not setting metadata
90 matching events found
No baseline correction applied

Loaded 

In [41]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [42]:
def train_evaluate_lda(
    X_train,
    y_train,
    X_test,
    y_test
):

    # Create LDA classifier
    lda = LinearDiscriminantAnalysis()

    # Train
    lda.fit(X_train, y_train)

    # Predictions
    y_train_pred = lda.predict(X_train)
    y_test_pred = lda.predict(X_test)

    # Accuracies
    train_accuracy = accuracy_score(
        y_train,
        y_train_pred
    )

    test_accuracy = accuracy_score(
        y_test,
        y_test_pred
    )

    # Test confusion matrix
    cm = confusion_matrix(
        y_test,
        y_test_pred
    )

    return (
        lda,
        y_train_pred,
        y_test_pred,
        train_accuracy,
        test_accuracy,
        cm
    )

In [43]:
lda_results = {}

for n_subjects in SUBJECT_EXPERIMENTS:

    print("\n" + "=" * 70)
    print(f"LDA Experiment : {n_subjects} Subjects")
    print("=" * 70)

    experiment_dir = os.path.join(
        FEATURE_DIR,
        f"{n_subjects}subjects"
    )

    # ----------------------------------
    # Load CSP features
    # ----------------------------------

    X_train_csp = np.load(
        os.path.join(experiment_dir, "X_train.npy")
    )

    y_train = np.load(
        os.path.join(experiment_dir, "y_train.npy")
    )

    X_test_csp = np.load(
        os.path.join(experiment_dir, "X_test.npy")
    )

    y_test = np.load(
        os.path.join(experiment_dir, "y_test.npy")
    )

    # ----------------------------------
    # Train and evaluate LDA
    # ----------------------------------

    (
        lda,
        y_train_pred,
        y_test_pred,
        train_accuracy,
        test_accuracy,
        cm
    ) = train_evaluate_lda(
        X_train_csp,
        y_train,
        X_test_csp,
        y_test
    )

    # ----------------------------------
    # Save results
    # ----------------------------------

    lda_results[n_subjects] = {
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
        "confusion_matrix": cm,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred
    }

    # ----------------------------------
    # Print results
    # ----------------------------------

    print(
        f"Train Accuracy : {train_accuracy:.4f} "
        f"({train_accuracy * 100:.2f}%)"
    )

    print(
        f"Test Accuracy  : {test_accuracy:.4f} "
        f"({test_accuracy * 100:.2f}%)"
    )

    print(
        f"Accuracy Gap   : "
        f"{(train_accuracy - test_accuracy) * 100:.2f}%"
    )

    print("\nTest Confusion Matrix")
    print(cm)

    print("\nTest Classification Report")

    print(
        classification_report(
            y_test,
            y_test_pred,
            target_names=["Left", "Right"],
            digits=4
        )
    )


LDA Experiment : 10 Subjects
Train Accuracy : 0.6254 (62.54%)
Test Accuracy  : 0.5259 (52.59%)
Accuracy Gap   : 9.95%

Test Confusion Matrix
[[39 31]
 [33 32]]

Test Classification Report
              precision    recall  f1-score   support

        Left     0.5417    0.5571    0.5493        70
       Right     0.5079    0.4923    0.5000        65

    accuracy                         0.5259       135
   macro avg     0.5248    0.5247    0.5246       135
weighted avg     0.5254    0.5259    0.5256       135


LDA Experiment : 20 Subjects
Train Accuracy : 0.5159 (51.59%)
Test Accuracy  : 0.5111 (51.11%)
Accuracy Gap   : 0.48%

Test Confusion Matrix
[[71 65]
 [67 67]]

Test Classification Report
              precision    recall  f1-score   support

        Left     0.5145    0.5221    0.5182       136
       Right     0.5076    0.5000    0.5038       134

    accuracy                         0.5111       270
   macro avg     0.5110    0.5110    0.5110       270
weighted avg     0.5111

In [44]:
print("\n" + "=" * 85)
print("CSP + LDA FINAL RESULTS")
print("=" * 85)

print(
    f"{'Subjects':<12}"
    f"{'Train Acc':<15}"
    f"{'Test Acc':<15}"
    f"{'Gap':<15}"
)

print("-" * 57)

for n_subjects in SUBJECT_EXPERIMENTS:

    train_acc = lda_results[
        n_subjects
    ]["train_accuracy"]

    test_acc = lda_results[
        n_subjects
    ]["test_accuracy"]

    gap = train_acc - test_acc

    print(
        f"{n_subjects:<12}"
        f"{train_acc * 100:<15.2f}"
        f"{test_acc * 100:<15.2f}"
        f"{gap * 100:<15.2f}"
    )


CSP + LDA FINAL RESULTS
Subjects    Train Acc      Test Acc       Gap            
---------------------------------------------------------
10          62.54          52.59          9.95           
20          51.59          51.11          0.48           
50          52.70          48.96          3.74           
109         51.36          50.56          0.79           
